# Notebook 00: Example 1, what makes a point anomalous

**Class 03_01** (anomaly detection I). The running example of the lecture, rebuilt so that every number on the
slides can be checked: one host's sessions described by two standardised features, $x_1$ = log bytes sent and
$x_2$ = log bytes received. They are strongly correlated ($\rho = 0.9$): a session that sends more also receives more.

* **Normal model:** $\mathbf{x}\sim\mathcal{N}(\boldsymbol\mu,C)$ with $\boldsymbol\mu=(0,0)$,
  $C=\begin{pmatrix}1&0.9\\0.9&1\end{pmatrix}$.
* **Point A** $=(2,2)$: a big but *consistent* session. **Point B** $=(1,-1)$: average sizes, but it sends a lot and
  receives little (the shape of an exfiltration).

Sections: A. z-score vs Mahalanobis; B. thresholds and the base rate; C. contamination and robust estimates (MCD);
D. distance, density and isolation on the same data (kNN, LOF, Isolation Forest).

In [ ]:
import sys

import numpy as np
import polars as pl
from matplotlib import pyplot as plt
from scipy import stats
from scipy.stats import chi2

sys.path[:0] = [".", "../../src"]  # this folder and the generic libraries of the repository
import aasplot as ap
import anomlib as an
from aasplot import BLUE, COL, FULL, GOLD, GREEN, GREY, LIGHT, RED

plt.rcParams["figure.dpi"] = 100
an.table_config()
ap.setup("03")
X = an.example1(500, seed=0)
A, B = an.EX1_POINTS["A"], an.EX1_POINTS["B"]
MU, COV = an.EX1_MU, an.EX1_COV
print("sample mean", X.mean(0).round(3), " sample correlation", np.corrcoef(X.T)[0, 1].round(3))

## A. z-score vs Mahalanobis distance

The **z-score** looks at one feature at a time: $z_j=(x_j-\mu_j)/\sigma_j$, and a point is anomalous when
$\max_j|z_j|$ is large. The **Mahalanobis distance** uses the covariance:
$d_M(\mathbf{x})^2=(\mathbf{x}-\boldsymbol\mu)^\top C^{-1}(\mathbf{x}-\boldsymbol\mu)$.
By hand with the true parameters: $C^{-1}=\frac{1}{0.19}\begin{pmatrix}1&-0.9\\-0.9&1\end{pmatrix}$, so
$d_M(A)^2=(4+4-7.2)/0.19=4.21$ and $d_M(B)^2=(1+1+1.8)/0.19=20.0$.

In [ ]:
Sinv = np.linalg.inv(an.EX1_COV)
rows = []
for name, p in [("A", A), ("B", B)]:
    d2 = float(p @ Sinv @ p)
    rows.append({"point": name, "max |z|": float(np.abs(p).max()), "Euclidean": float(np.linalg.norm(p)),
                 "Mahalanobis^2": round(d2, 2), "Mahalanobis": round(np.sqrt(d2), 2),
                 "P(chi2_2 > d^2)": float(chi2.sf(d2, 2))})  # fmt: skip
print(pl.DataFrame(rows))
print(f"det C = {np.linalg.det(an.EX1_COV):.2f}; chi2 tail of d^2 = 20: {chi2.sf(20, 2):.6f} (= e^-10); "
      f"99% ellipse d_M = {np.sqrt(chi2.ppf(0.99, 2)):.2f}")  # fmt: skip

Under the Gaussian model $d_M^2$ follows a $\chi^2$ distribution with $p=2$ degrees of freedom, so the tail
probability is a calibrated *p-value*: B is a 1-in-22,000 event, A a 1-in-8 event. The z-score ranks them the other
way round: it flags A (|z| = 2) and calls B (|z| = 1) perfectly normal.

In [ ]:
g = np.linspace(-4, 4, 200)
G = np.stack(np.meshgrid(g, g), -1).reshape(-1, 2)
dm = np.sqrt(np.einsum("ij,jk,ik->i", G, Sinv, G)).reshape(200, 200)
fig, ax = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True)
for a, (Z, title) in zip(ax, [(np.abs(G).max(1).reshape(200, 200), "max |z| = 2, 3"), (dm, "Mahalanobis = 2, 3")]):
    a.scatter(*X.T, s=6, alpha=0.4)
    a.contour(g, g, Z, levels=[2, 3], colors=["tab:orange", "tab:red"])
    for n, p in [("A", A), ("B", B)]:
        a.plot(*p, "k*", ms=12)
        a.annotate(n, p + 0.15)
    a.set_title(title)
    a.set_xlabel("$x_1$ (log bytes sent, std.)")
ax[0].set_ylabel("$x_2$ (log bytes received, std.)")
plt.tight_layout()
plt.show()

## B. Thresholds and the base rate

A detector needs a **threshold** $t$: alarm when $s(\mathbf{x})>t$. Without labelled attacks we set it on normal
data only, as the $(1-\alpha)$ quantile of the scores, which gives a **false-positive rate** of about $\alpha$.
For the Gaussian model there is a closed form, $t=\chi^2_{2,1-\alpha}$.

In [ ]:
for alpha in [0.05, 0.01, 0.001]:
    s = np.einsum("ij,jk,ik->i", X, Sinv, X)
    print(
        f"alpha={alpha:<6} chi2 quantile={chi2.ppf(1 - alpha, 2):6.2f}  empirical quantile of 500 normals={np.quantile(s, 1 - alpha):6.2f}"
    )

**Base rate.** Even a good detector drowns in false alarms when attacks are rare (Axelsson 2000). Bayes' rule gives
the share of alarms that are real, $P(\text{attack}\mid\text{alarm})=\frac{\text{TPR}\,b}{\text{TPR}\,b+\text{FPR}\,(1-b)}$,
with $b$ the share of attacks among events.

In [ ]:
rows = []
for b in [0.1, 0.01, 0.001, 0.0001]:
    rows.append(
        {"base rate b": b, **{f"FPR={f}": round(an.precision_at_base_rate(0.95, f, b), 4) for f in [0.01, 0.001]}}
    )
print("precision P(attack | alarm) for TPR = 0.95")
print(pl.DataFrame(rows))
print("alarms per day at 1,000,000 events, FPR 0.01:", int(0.01 * 1_000_000))

## C. Contamination: the estimate is part of the attack surface

The mean and covariance are *estimated* from data that may already contain attacks. Add 5% of points around B to
the training data: the covariance stretches towards B and its distance shrinks. The **Minimum Covariance
Determinant** (MCD, Rousseeuw 1984) fits the covariance on the $h$ most concentrated points only, so a minority of
outliers cannot move it.

In [ ]:
rng = np.random.default_rng(1)
rows = []
for frac in [0.0, 0.05, 0.10, 0.20]:
    k = int(frac * len(X))
    Xc = np.vstack([X, B * 1.5 + rng.normal(0, 0.2, (k, 2))])
    for det in [an.Mahalanobis(), an.Mahalanobis(robust=True)]:
        det.fit(Xc)
        rows.append({"contamination": frac, "estimator": det.name, "d^2(B)": round(float(det.score(B[None])[0]), 2),
                     "d^2(A)": round(float(det.score(A[None])[0]), 2)})  # fmt: skip
print(pl.DataFrame(rows))

Ledoit-Wolf (the plain estimator here) is pulled towards B as soon as it is in the data; MCD keeps $d^2(B)$ high
until the contamination approaches its breakdown point (here half of the data minus the dimension).

## D. Distance, density and isolation

Three non-Gaussian views of the same question:
* **kNN distance:** the distance to the $k$-th nearest normal point (large = isolated).
* **LOF** (Breunig et al. 2000): the local density of a point divided by that of its neighbours (about 1 = normal).
* **Isolation Forest** (Liu et al. 2008): random axis-aligned cuts; a point that is isolated after few cuts is
  anomalous. Score $s(\mathbf{x})=2^{-E[h(\mathbf{x})]/c(n)}$ with $h$ the path length and
  $c(n)=2H(n-1)-2(n-1)/n$ the average path of an unsuccessful search in a binary tree ($H(i)\approx\ln i+0.5772$).

In [ ]:
def c(n: int) -> float:
    return 2 * (np.log(n - 1) + 0.5772156649) - 2 * (n - 1) / n


print(f"Euler's constant {np.euler_gamma:.4f}; c(256) = {c(256):.3f}: a point isolated after 4 cuts scores 2^(-4/c) = {2 ** (-4 / c(256)):.3f};"
      f" after c(256) cuts it scores 0.5")  # fmt: skip
rows = []
for det in [an.KNNDist(k=5), an.LOF(k=20), an.IForest(), an.Mahalanobis()]:
    det.fit(X)
    s = det.score(np.vstack([X, A, B]))
    rank = lambda v: int((s[:-2] >= v).sum())  # noqa: E731  (how many normal points score at least as high)
    rows.append({"detector": det.name, "normals above A": rank(s[-2]), "normals above B": rank(s[-1])})
print("rank of A and B among the 500 normal points (0 = more anomalous than every normal point)")
print(pl.DataFrame(rows))

B lies in an *empty* region between normal points, so the methods that look at local distances or densities
(kNN, LOF) and the covariance (Mahalanobis) put it above every or almost every normal point. The Isolation Forest
cuts along the axes only: B has ordinary values on each axis and needs more cuts than the corner points of the
cloud (31 normals rank higher). Axis-aligned cuts are blind to correlations; the *extended* Isolation Forest
(Hariri et al. 2021) uses random oblique cuts for this reason.

**LOF and dense attacks.** A flood (thousands of identical connections) forms a *dense* cluster: its points have
neighbours as dense as themselves, so their LOF is about 1. Notebook 02 shows that LOF misses most of the DoS rows
of KDD99 for this reason.

In [ ]:
flood = np.array([3.0, -0.5]) + rng.normal(0, 0.02, (60, 2))
lof = an.LOF(k=20).fit(X)
knn = an.KNNDist(k=5).fit(X)
print(f"flood of 60 near-identical points: mean LOF score {lof.score(flood).mean():.2f} (normal median "
      f"{np.median(lof.score(X)):.2f}), kNN distance {knn.score(flood).mean():.2f} (normal median {np.median(knn.score(X)):.2f})")  # fmt: skip
lof_in = an.LOF(k=20).fit(np.vstack([X, flood]))
print(f"if the flood is in the training data (unsupervised setting): LOF of the flood {lof_in.score(flood).mean():.2f}")

## E. The figures of the slides

Every figure of the class 03_01 slides is drawn in a cell of these notebooks: the cell shows it here and stores it
as a transparent PDF in `assets/figures/plots/03/`, the file the slide includes. This notebook draws the
illustrations on Example 1 and on small toy data; notebooks 01-03 draw the results on KDD99. The first group
covers Example 1 and the distribution-based family.

In [ ]:
def kinds() -> None:
    rng = np.random.default_rng(1)
    fig, axs = plt.subplots(1, 3, figsize=(FULL, 1.15), gridspec_kw={"wspace": 0.28})
    c = rng.normal(0, 0.6, (70, 2))
    ax = axs[0]
    ax.scatter(c[:, 0], c[:, 1], s=5, color=GREY, alpha=0.7, lw=0)
    ax.scatter([2.6], [2.4], s=24, color=RED)
    ax.set(xlabel="upload (MB)", ylabel="duration", title="point", xticks=[], yticks=[])
    h = np.arange(72)
    y = 55 + 40 * np.sin(2 * np.pi * (h - 9) / 24) + rng.normal(0, 3, h.size)
    y[51] = 78
    ax = axs[1]
    ax.plot(h, y, color=GREY, lw=0.9)
    ax.axhline(78, color=RED, ls="--", lw=0.7)
    ax.scatter([51], [78], s=24, color=RED, zorder=5)
    ax.set(xlabel="hour", ylabel="logins per minute", title="contextual", xticks=[0, 24, 48, 72], yticks=[])
    t = np.arange(120)
    s = np.sin(2 * np.pi * t / 20) + rng.normal(0, 0.05, t.size)
    s[60:80] = 0.0
    ax = axs[2]
    ax.plot(t, s, color=GREY, lw=0.9)
    ax.plot(t[60:81], s[60:81], color=RED, lw=2)
    ax.set(xlabel="time", ylabel="sensor", title="collective", xticks=[], yticks=[])
    ap.save(fig, "kinds")


kinds()

In [ ]:
from sklearn.svm import SVC


def settings() -> None:
    """The same events with different labels: what each setting learns (a border, a region, a distorted region)."""
    rng = np.random.default_rng(4)
    c = rng.normal(0, 0.6, (60, 2))
    a = np.array([[2.3, 2.0], [2.6, 1.0], [-2.2, 2.3], [2.0, -2.2], [-2.5, -1.8], [0.3, 2.8], [-1.0, -2.6]])
    fig, axs = plt.subplots(1, 3, figsize=(FULL, 1.3), gridspec_kw={"wspace": 0.2})
    for ax in axs:
        ax.set(xlim=(-3.2, 3.2), ylim=(-3.2, 3.2), xticks=[], yticks=[], aspect="equal")
        ax.grid(False)
        for sp in ax.spines.values():
            sp.set_visible(True)
            sp.set_color(LIGHT)
    gx, gy = np.meshgrid(np.linspace(-3.2, 3.2, 200), np.linspace(-3.2, 3.2, 200))
    grid = np.c_[gx.ravel(), gy.ravel()]
    ax = axs[0]  # supervised: normal and attacks, both labelled; learn the border between them
    ax.scatter(c[:, 0], c[:, 1], s=7, color=BLUE, lw=0)
    ax.scatter(a[:, 0], a[:, 1], s=16, color=RED, marker="^", lw=0)
    clf = SVC(kernel="rbf", gamma=0.4, C=10).fit(np.r_[c, a], np.r_[np.zeros(len(c)), np.ones(len(a))])
    ax.contour(gx, gy, clf.decision_function(grid).reshape(gx.shape), levels=[0], colors=[GREEN], linewidths=1.3)
    ax.set_title("supervised", weight="bold")
    ax.set_xlabel("labelled normal + attacks:\nlearns the border", fontsize=7)
    ax = axs[1]  # semi-supervised: only normal events; learn the region of normal
    ax.scatter(c[:, 0], c[:, 1], s=7, color=BLUE, lw=0)
    mu, cov = c.mean(0), np.cov(c.T)
    ax.plot(*ap.ellipse(mu, cov, 3.03).T, color=GREEN, lw=1.3)
    ax.set_title("semi-supervised", weight="bold")
    ax.set_xlabel("normal only:\nlearns what normal is", fontsize=7)
    ax = axs[2]  # unsupervised: unlabelled mix; attacks stretch the region of normal
    allx = np.r_[c, a]
    ax.scatter(allx[:, 0], allx[:, 1], s=7, color=LIGHT, lw=0)
    ax.plot(*ap.ellipse(allx.mean(0), np.cov(allx.T), 2.0).T, color=GREEN, lw=1.3)
    ax.set_title("unsupervised", weight="bold")
    ax.set_xlabel("unlabelled, attacks inside:\nregion of normal stretched", fontsize=7)
    fig.legend(handles=[plt.Line2D([], [], marker="o", color=BLUE, ls="", ms=3.5, label="normal"),
                        plt.Line2D([], [], marker="^", color=RED, ls="", ms=4, label="attack"),
                        plt.Line2D([], [], marker="o", color=LIGHT, ls="", ms=3.5, label="unlabelled"),
                        plt.Line2D([], [], color=GREEN, lw=1.3, label="what the detector learns")],
               loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.16), fontsize=7, handletextpad=0.3, columnspacing=1.2)  # fmt: skip
    fig.subplots_adjust(bottom=0.36)
    ap.save(fig, "settings")


settings()

In [ ]:
def dist1d() -> None:
    x = np.linspace(-4, 4, 400)
    fig, ax = plt.subplots(figsize=(FULL, 1.1))
    ax.plot(x, stats.norm.pdf(x), color=BLUE, lw=1.5)
    m = x >= 2
    ax.fill_between(x[m], stats.norm.pdf(x[m]), color=RED, alpha=0.3, lw=0)
    ax.scatter([0.4, -0.7, 1.0, -0.1, 0.9, -1.5], np.zeros(6) + 0.008, s=9, color=GREY, zorder=4)
    ax.scatter([3.1], [0.008], s=22, color=RED, zorder=5)
    ax.annotate(
        "rare",
        (3.1, 0.02),
        (3.1, 0.12),
        color=RED,
        ha="center",
        arrowprops={"arrowstyle": "-", "color": RED, "lw": 0.6},
    )
    ax.text(2.35, 0.065, "$P=2.3\\%$", color=RED, ha="left")
    ax.text(0, 0.43, "normal values", color=BLUE, ha="center")
    ax.set(xlabel="feature value $x$", ylabel="density", yticks=[], xticks=[-2, 0, 2], xlim=(-4, 4), ylim=(0, 0.47))
    ax.grid(False)
    ap.save(fig, "dist1d")


dist1d()

In [ ]:
def cloud(ax, n: int = 500) -> None:
    ax.scatter(X[:n, 0], X[:n, 1], s=3, color=GREY, alpha=0.55, lw=0, zorder=2)


def points_ab(ax, label: bool = True) -> None:
    ax.scatter(*A, s=22, color=RED, zorder=5)
    ax.scatter(*B, s=22, color=BLUE, marker="s", zorder=5)
    if label:
        ax.annotate("A", A, (A[0] - 0.1, A[1] + 0.35), color=RED, weight="bold", ha="center")
        ax.annotate("B", B, (B[0] + 0.45, B[1] - 0.15), color=BLUE, weight="bold", ha="center")


def square_axes(ax, lim: float = 3.5, ticks=(-2, 0, 2)) -> None:
    """Equal-scale square axes of Example 1 (standardised log bytes sent and received)."""
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xlabel("$x_1$ log bytes sent")
    ax.set_ylabel("$x_2$ log bytes received")


def ex1_data() -> None:
    fig, ax = plt.subplots(figsize=(COL, COL))
    cloud(ax)
    points_ab(ax)
    square_axes(ax)
    ap.save(fig, "ex1_data")


ex1_data()

In [ ]:
def ex1_z() -> None:
    fig, ax = plt.subplots(figsize=(COL, COL))
    cloud(ax)
    ax.plot([-2, 2, 2, -2, -2], [-2, -2, 2, 2, -2], color=GOLD, ls="--", lw=1.3)
    points_ab(ax)
    ax.text(0, -3.1, "box: $|z_j|\\leq 2$ in both features", color=GOLD, ha="center", va="center", fontsize=8)
    square_axes(ax)
    ap.save(fig, "ex1_z")


ex1_z()

In [ ]:
def ex1_ellipse() -> None:
    fig, ax = plt.subplots(figsize=(COL, COL))
    cloud(ax)
    ax.plot(*ap.ellipse(MU, COV, 3.03).T, color=BLUE, lw=1.3)
    points_ab(ax)
    ax.text(-3.3, 3.2, "ellipse: 99% of normal sessions", color=BLUE, ha="left", va="top", fontsize=7)
    square_axes(ax)
    ap.save(fig, "ex1_ellipse")


ex1_ellipse()

In [ ]:
P = np.linalg.inv(COV)


def d2(x) -> float:
    return float((x - MU) @ P @ (x - MU))


def ex1_levels() -> None:
    fig, ax = plt.subplots(figsize=(COL, COL))
    cloud(ax)
    ax.plot(*ap.ellipse(MU, COV, d2(A) ** 0.5).T, color=RED, lw=1.3)
    ax.plot(*ap.ellipse(MU, COV, d2(B) ** 0.5).T, color=BLUE, lw=1.3)
    ax.text(-3.3, 3.2, "red: ellipse through A", color=RED, ha="left", va="top", fontsize=7)
    ax.text(-3.3, 2.65, "blue: ellipse through B", color=BLUE, ha="left", va="top", fontsize=7)
    points_ab(ax)
    square_axes(ax)
    ap.save(fig, "ex1_levels")
    print("d2 A, B:", round(d2(A), 2), round(d2(B), 2))


ex1_levels()

In [ ]:
import seaborn as sns


def threshold() -> None:
    s = np.array([d2(x) for x in X])
    fig, ax = plt.subplots(figsize=(FULL, 1.35))
    sns.histplot(s, bins=np.linspace(0, 16, 25), stat="density", color=LIGHT, edgecolor="white", linewidth=0.4, ax=ax)
    xx = np.linspace(0, 16, 200)
    ax.plot(xx, np.exp(-xx / 2) / 2, color=BLUE, lw=1.4, label="$\\chi^2_2$ density")
    for (a, q), col, y in zip(((0.05, 5.99), (0.01, 9.21), (0.001, 13.82)), (GOLD, RED, "#7B241C"), (0.30, 0.25, 0.20)):
        ax.axvline(q, color=col, ls="--", lw=1)
        ax.text(q + 0.2, y, f"$\\alpha={a}$", color=col, fontsize=8)
    ax.set(xlabel="score $d_M^2$ of 500 normal sessions", ylabel="density", yticks=[], xlim=(0, 16), ylim=(0, 0.5))
    ax.legend(loc="upper right")
    ax.grid(False)
    ap.save(fig, "threshold")
    print("empirical quantiles:", [round(float(np.quantile(s, 1 - a)), 2) for a in (0.05, 0.01, 0.001)])


threshold()

In [ ]:
def baserate() -> None:
    b = np.logspace(-5, -1, 60)
    tpr = 0.95
    fig, ax = plt.subplots(figsize=(FULL, 1.0))
    for fpr, col in ((0.01, RED), (0.001, BLUE)):
        ax.plot(b, tpr * b / (tpr * b + fpr * (1 - b)), color=col, lw=1.5, label=f"FPR {fpr:.1%}".replace(".0%", "%"))
    ax.scatter([1e-4, 1e-4], [0.0094, 0.087], s=14, color="black", zorder=5)
    ax.set(xscale="log", xlabel="base rate $b$", ylabel="$P(\\mathrm{attack}\\mid\\mathrm{alarm})$", ylim=(0, 1))
    ax.legend(loc="upper left")
    ap.save(fig, "baserate")


baserate()

In [ ]:
def mcd() -> None:
    """Notebook 00 part C: copies of a B-like session in the training rows; Ledoit-Wolf against MCD."""
    rng = np.random.default_rng(1)
    fr, lw, mc = [0.0, 0.05, 0.10, 0.20], [], []
    for frac in fr:
        k = int(frac * len(X))
        Xc = np.vstack([X, B * 1.5 + rng.normal(0, 0.2, (k, 2))])
        lw.append(float(an.Mahalanobis().fit(Xc).score(B[None])[0]))
        mc.append(float(an.Mahalanobis(robust=True).fit(Xc).score(B[None])[0]))
    pc = [100 * f for f in fr]
    fig, ax = plt.subplots(figsize=(FULL * 0.8, 1.35))
    ax.plot(pc, lw, color=RED, marker="o", ms=3.5, label="Ledoit-Wolf")
    ax.plot(pc, mc, color=BLUE, marker="s", ms=3.5, label="MCD")
    ax.axhline(9.21, color=GREY, ls="--", lw=0.8)
    ax.text(0.3, 10.3, "threshold $t=9.21$ (1% false alarms)", color=GREY, ha="left", fontsize=7)
    ax.set(xlabel="attacks mixed into the training rows (%)", ylabel="$d_M^2(B)$", xticks=pc, ylim=(0, 36))
    ax.legend(loc="upper right", ncol=2)
    ap.save(fig, "mcd")
    print("mcd LW", np.round(lw, 1), "MCD", np.round(mc, 1))


mcd()

The second group: one toy picture per detector family (distance, density, isolation, boundary, reconstruction,
clusters). The numbers printed with them are the ones quoted on the slides.

In [ ]:
from sklearn.neighbors import NearestNeighbors


def circle(c, r: float, n: int = 120) -> np.ndarray:
    t = np.linspace(0, 2 * np.pi, n)
    return np.stack([c[0] + r * np.cos(t), c[1] + r * np.sin(t)], 1)


def knn() -> None:
    nn = NearestNeighbors(n_neighbors=5).fit(X)
    fig, ax = plt.subplots(figsize=(COL, COL))
    cloud(ax)
    for name, q, col in (("N", np.array([0.3, 0.5]), GREEN), ("A", A, RED), ("B", B, BLUE)):
        d, idx = nn.kneighbors(q[None])
        r = float(d[0, -1])
        ax.plot(*circle(q, r).T, color=col, lw=1.2)
        ax.scatter(X[idx[0], 0], X[idx[0], 1], s=7, color=col, zorder=4)
        ax.scatter(*q, s=22, color=col, zorder=5, marker="s" if name == "B" else "o")
        tx = {"N": (-3.3, 2.6), "A": (1.9, 3.2), "B": (1.6, -3.2)}[name]
        ax.annotate(
            f"{name}: {r:.2f}",
            q,
            tx,
            color=col,
            fontsize=7,
            weight="bold",
            ha="left",
            va="center",
            arrowprops={"arrowstyle": "-", "color": col, "lw": 0.6},
            bbox={"fc": "white", "ec": "none", "alpha": 0.8, "pad": 1},
        )
        print("kNN", name, round(r, 2))
    square_axes(ax)
    ap.save(fig, "knn")


knn()

In [ ]:
from sklearn.neighbors import LocalOutlierFactor


def lof_toy(seed: int = 3):
    rng = np.random.default_rng(seed)
    return np.r_[rng.normal([-2.4, 0], 0.22, (60, 2)), rng.normal([2.2, 0], 0.9, (60, 2))]


def lof() -> None:
    Xt = lof_toy()
    nn = NearestNeighbors(n_neighbors=5).fit(Xt)
    model = LocalOutlierFactor(n_neighbors=5, novelty=True).fit(Xt)
    P = np.array([-1.35, 0.35])
    rp = float(nn.kneighbors(P[None])[0][0, -1])
    cand = [
        (abs(float(nn.kneighbors(np.array([[x, y]]))[0][0, -1]) - rp), x, y)
        for x in np.linspace(1.5, 4.5, 31)
        for y in np.linspace(-2.5, 2.5, 41)
    ]
    Q = np.array(min(cand)[1:])
    fig, ax = plt.subplots(figsize=(FULL, 1.4))
    ax.scatter(Xt[:, 0], Xt[:, 1], s=5, color=GREY, alpha=0.7, lw=0)
    for name, q, col in (("P", P, RED), ("Q", Q, BLUE)):
        r = float(nn.kneighbors(q[None])[0][0, -1])
        lo = float(-model.score_samples(q[None])[0])
        ax.plot(*circle(q, r).T, color=col, lw=1.2)
        ax.scatter(*q, s=22, color=col, zorder=5)
        ax.annotate(f"{name}: LOF {lo:.1f}", q, (q[0], q[1] - r - 0.5), color=col, ha="center", weight="bold")
        print("LOF", name, "kNN", round(r, 2), "LOF", round(lo, 2))
    ax.text(-2.4, 0.9, "dense", color=GREY, ha="center")
    ax.text(2.2, 2.2, "sparse", color=GREY, ha="center")
    ax.set(xlim=(-4, 5), ylim=(-3.2, 3), aspect="equal", xticks=[], yticks=[])
    ax.grid(False)
    ap.save(fig, "lof")


lof()

In [ ]:
def lof_flood() -> None:
    rng = np.random.default_rng(0)  # notebook 00 part D: 60 near-identical events
    flood = np.array([3.0, -0.5]) + rng.normal(0, 0.02, (60, 2))
    fig, ax = plt.subplots(figsize=(COL, COL))
    cloud(ax)
    ax.scatter(flood[:, 0], flood[:, 1], s=7, color=RED, zorder=4)
    ax.annotate(
        "60 identical events",
        (3.0, -0.5),
        (0.6, -2.7),
        color=RED,
        fontsize=7,
        arrowprops={"arrowstyle": "-", "color": RED, "lw": 0.6},
    )
    square_axes(ax)
    ap.save(fig, "lof_flood")


lof_flood()

In [ ]:
def isolate(Xs, target, rng, lo, hi):
    """Random isolation tree path of one point: the cuts (axis, value, cell lo, cell hi)."""
    cuts, idx = [], np.arange(len(Xs))
    lo, hi = lo.copy(), hi.copy()
    while len(idx) > 1:
        side = Xs[idx]
        a = rng.integers(2)
        cut = rng.uniform(side[:, a].min(), side[:, a].max())
        cuts.append((a, cut, lo.copy(), hi.copy()))
        keep = (Xs[:, a] <= cut) == (target[a] <= cut)
        idx = idx[keep[idx]]
        if target[a] <= cut:
            hi[a] = cut
        else:
            lo[a] = cut
    return cuts


def iso_cuts() -> None:
    rng = np.random.default_rng(5)
    Xs = np.r_[rng.normal(0, 0.8, (40, 2)), [[3.4, 2.7]]]
    lo, hi = Xs.min(0) - 0.2, Xs.max(0) + 0.2
    fig, axs = plt.subplots(1, 2, figsize=(FULL, 1.55), gridspec_kw={"wspace": 0.1})
    for k, (name, target, ok) in enumerate(
        (("anomaly", Xs[-1], lambda n: n <= 4), ("normal point", Xs[3], lambda n: n >= 9))
    ):
        for s in range(200):
            cuts = isolate(Xs, target, np.random.default_rng(s), lo, hi)
            if ok(len(cuts)):
                break
        ax = axs[k]
        ax.scatter(Xs[:-1, 0], Xs[:-1, 1], s=6, color=GREY, lw=0)
        ax.scatter(*Xs[-1], s=24, color=RED, zorder=5)
        if k:
            ax.scatter(*target, s=24, color=BLUE, marker="s", zorder=5)
        for a, cut, cl, ch in cuts:
            ax.plot(
                [cut, cut] if a == 0 else [cl[0], ch[0]], [cl[1], ch[1]] if a == 0 else [cut, cut], color=GOLD, lw=1
            )
        ax.set(xlim=(lo[0], hi[0]), ylim=(lo[1], hi[1]), xticks=[], yticks=[], title=f"{name}: {len(cuts)} cuts")
        ax.grid(False)
        print("iso", name, len(cuts))
    ap.save(fig, "iso_cuts")


iso_cuts()

In [ ]:
def iso_axis() -> None:
    w = 0.15
    fig, ax = plt.subplots(figsize=(COL, COL))
    ax.axvspan(B[0] - w, B[0] + w, color=GOLD, alpha=0.25, lw=0)
    ax.axhspan(B[1] - w, B[1] + w, color=GOLD, alpha=0.25, lw=0)
    cloud(ax)
    points_ab(ax)
    square_axes(ax)
    ap.save(fig, "iso_axis")
    print("strips:", int((abs(X[:, 0] - B[0]) < w).sum()), int((abs(X[:, 1] - B[1]) < w).sum()))


iso_axis()

In [ ]:
from sklearn.svm import OneClassSVM


def ocsvm() -> None:
    m = OneClassSVM(nu=0.05, gamma="scale").fit(X)
    g = np.linspace(-3.5, 3.5, 200)
    gx, gy = np.meshgrid(g, g)
    z = m.decision_function(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    out = m.decision_function(X) < 0
    fig, ax = plt.subplots(figsize=(COL, COL))
    ax.scatter(X[~out, 0], X[~out, 1], s=3, color=GREY, alpha=0.55, lw=0)
    ax.scatter(X[out, 0], X[out, 1], s=6, color=RED, lw=0, zorder=4)
    ax.contour(gx, gy, z, levels=[0], colors=[BLUE], linewidths=1.4)
    points_ab(ax)
    square_axes(ax)
    ap.save(fig, "ocsvm")
    print(
        "ocsvm outside",
        int(out.sum()),
        "A",
        m.decision_function(A[None]).round(2),
        "B",
        m.decision_function(B[None]).round(2),
    )


ocsvm()

In [ ]:
from sklearn.decomposition import PCA


def pca() -> None:
    p = PCA(1).fit(X)
    v = p.components_[0]
    fig, ax = plt.subplots(figsize=(COL, COL))
    cloud(ax)
    t = np.array([-3.4, 3.4])[:, None] * v + p.mean_
    ax.plot(t[:, 0], t[:, 1], color=BLUE, lw=1.4)
    for q, col in ((A, RED), (B, BLUE)):
        f = p.mean_ + ((q - p.mean_) @ v) * v
        ax.plot([q[0], f[0]], [q[1], f[1]], color=col, ls="--", lw=1.3)
        ax.scatter(*f, s=14, facecolors="none", edgecolors=col, zorder=4)
        print("residual", round(float(np.sum((q - f) ** 2)), 2))
    points_ab(ax)
    ax.text(-3.3, 3.0, "first direction", color=BLUE, fontsize=7)
    square_axes(ax)
    ap.save(fig, "pca")


pca()

In [ ]:
from pynnmf import nmf_hals
from sklearn.decomposition import FastICA


def bss_toy() -> None:
    """Three decompositions of the same non-Gaussian, non-negative data: PCA axes, ICA directions, NMF parts."""
    rng = np.random.default_rng(2)
    S = rng.uniform(0, 1, (400, 2))
    M = np.array([[1.0, 0.25], [0.35, 0.9]])  # true mixing: rows are the two parts
    Xm = S @ M
    pc = PCA(2).fit(Xm)
    ic = FastICA(2, whiten="unit-variance", random_state=0, max_iter=1000).fit(Xm)
    _, _, h, _ = nmf_hals(Xm, 2, n=300, seed=0, init="nndsvd")
    mean = Xm.mean(0)
    panels = (("PCA", [(mean, c * np.sqrt(ev) * 2) for c, ev in zip(pc.components_, pc.explained_variance_)]),
              ("ICA", [(mean, c / np.linalg.norm(c) * 0.6) for c in ic.mixing_.T]),
              ("NMF", [(np.zeros(2), r / np.linalg.norm(r) * 1.1) for r in h]))  # fmt: skip
    fig, axs = plt.subplots(1, 3, figsize=(FULL, 1.5), gridspec_kw={"wspace": 0.15})
    caps = (
        "orthogonal axes,\nmost variance first",
        "independent sources,\nnon-orthogonal directions",
        "non-negative parts\nthat add up",
    )
    for ax, (title, vecs), caption in zip(axs, panels, caps):
        ax.scatter(Xm[:, 0], Xm[:, 1], s=3, color=LIGHT, lw=0)
        for (o, v), col in zip(vecs, (BLUE, RED)):
            ax.annotate("", o + v, o, arrowprops={"arrowstyle": "-|>", "color": col, "lw": 1.5})
        ax.set(xlim=(-0.15, 1.5), ylim=(-0.15, 1.5), aspect="equal", xticks=[], yticks=[], title=title)
        ax.set_xlabel(caption, fontsize=7)
        ax.title.set_weight("bold")
        ax.grid(False)
    ap.save(fig, "bss_toy")


bss_toy()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture


def toy_profiles(seed: int = 2):
    rng = np.random.default_rng(seed)
    cen = np.array([[-2.6, 1.6], [2.6, 1.8], [0.0, -2.3]])
    return np.concatenate([rng.normal(c, 0.55, (70, 2)) for c in cen])


def kmeans_gmm() -> None:
    Xp = toy_profiles()
    far, between = np.array([4.0, -2.8]), np.array([0.1, 1.0])
    km = KMeans(3, n_init=10, random_state=0).fit(Xp)
    gm = GaussianMixture(3, random_state=0).fit(Xp)
    fig, axs = plt.subplots(1, 2, figsize=(FULL, 1.7), gridspec_kw={"wspace": 0.08})
    for ax in axs:
        ax.scatter(Xp[:, 0], Xp[:, 1], s=4, color=GREY, alpha=0.7, lw=0)
        ax.scatter(*far, s=22, color=RED, zorder=5)
        ax.scatter(*between, s=22, color=RED, zorder=5)
        ax.set(xlim=(-4.5, 5), ylim=(-4, 3.5), aspect="equal", xticks=[], yticks=[])
        ax.grid(False)
    ax = axs[0]
    ax.set_title("k-means")
    ax.title.set_fontsize(7.5)
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1], marker="x", s=40, color=BLUE, lw=2, zorder=5)
    for q in (far, between):
        c = km.cluster_centers_[km.predict(q[None])[0]]
        ax.plot([q[0], c[0]], [q[1], c[1]], color=RED, ls="--", lw=1)
        print("kmeans dist", round(float(np.linalg.norm(q - c)), 2))
    ax = axs[1]
    ax.set_title("Gaussian mixture")
    ax.title.set_fontsize(7.5)
    for mu, cov in zip(gm.means_, gm.covariances_):
        ax.plot(*ap.ellipse(mu, cov, 3.03).T, color=BLUE, lw=1.2)
    print(
        "gmm -log p",
        (-gm.score_samples(np.array([far, between]))).round(1),
        "typical",
        np.median(-gm.score_samples(Xp)).round(1),
    )
    ap.save(fig, "kmeans_gmm")


kmeans_gmm()

In [ ]:
from sklearn.cluster import DBSCAN


def dbscan() -> None:
    Xp = toy_profiles()
    noise = np.random.default_rng(9).uniform([-4, -3.5], [4.5, 3.2], (7, 2))
    Z = np.r_[Xp, noise]
    lab = DBSCAN(eps=0.6, min_samples=6).fit_predict(Z)
    fig, ax = plt.subplots(figsize=(COL, COL))
    for k, (col, mk) in enumerate(((BLUE, "o"), (GREEN, "s"), (GOLD, "^"))):
        ax.scatter(Z[lab == k, 0], Z[lab == k, 1], s=6, color=col, marker=mk, lw=0)
    ax.scatter(Z[lab == -1, 0], Z[lab == -1, 1], s=26, color=RED, marker="x", lw=1.6, zorder=5)
    ax.plot(*circle(Z[lab == -1][0], 0.6).T, color=RED, ls="--", lw=1)
    ax.set(xlim=(-4.5, 5), ylim=(-4, 3.5), aspect="equal", xticks=[], yticks=[])
    ax.grid(False)
    ap.save(fig, "dbscan")
    print("dbscan noise", int((lab == -1).sum()), "clusters", sorted(set(lab.tolist())))


dbscan()